## Import Libraries

In [1]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, Subset
import pickle
from PIL import Image
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

## Configuration

In [2]:
MODEL_NAME = 'resnet50'
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS_INITIAL = 10  # Initial epochs with frozen base
NUM_EPOCHS_FINE = 50    # Fine-tuning epochs with unfrozen layers
BATCH_DIR = r"D:\AIoT_project\data_sets\miniimagenet_batches"  # Path to batch files
SAVE_PATH = r"D:\AIoT_project\models\resnet50_miniimagenet_90_v2.pth"  # Save path

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Dataset Preparation

In [3]:
class MiniImageNetDataset(Dataset):
    def __init__(self, batch_dir, transform=None):
        self.batch_dir = batch_dir
        self.transform = transform
        self.images = []
        self.labels = []

        # Load metadata
        meta_file = os.path.join(batch_dir, "batches.meta")
        with open(meta_file, "rb") as f:
            meta = pickle.load(f)
        self.classes = meta["label_names"]

        # Load batch files
        batch_files = [f for f in os.listdir(batch_dir) if f.startswith("data_batch_") and f.endswith(".pkl")]
        for batch_file in batch_files:
            with open(os.path.join(batch_dir, batch_file), "rb") as f:
                batch_data = pickle.load(f)
                self.images.extend(batch_data["images"])
                self.labels.extend(batch_data["labels"])

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.uint8(image))
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.images)

# Training transformations with augmentation
train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transformations (no augmentation)
val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load and split dataset
full_dataset = MiniImageNetDataset(batch_dir=BATCH_DIR)
num_samples = len(full_dataset)
indices = list(range(num_samples))
np.random.shuffle(indices)
train_size = int(0.8 * num_samples)
train_indices, val_indices = indices[:train_size], indices[train_size:]

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
num_classes = len(full_dataset.classes)
print(f"Loaded Mini-ImageNet: {len(train_dataset)} train, {len(val_dataset)} val, {num_classes} classes.")

Loaded Mini-ImageNet: 40000 train, 10000 val, 100 classes.


## Model Definition

In [4]:
class FineTuneResNet50(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.2):
        super(FineTuneResNet50, self).__init__()
        self.base_model = models.resnet50(pretrained=True)
        
        # Freeze base layers
        for param in self.base_model.parameters():
            param.requires_grad = False
        
        # Replace fc layer
        in_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.base_model(x)
    
    def unfreeze_layers(self, num_layers_to_unfreeze=20):
        children = list(self.base_model.children())
        total_layers = len(children)
        for child in children[max(0, total_layers - num_layers_to_unfreeze):]:
            for param in child.parameters():
                param.requires_grad = True

# Initialize model
model = FineTuneResNet50(num_classes=num_classes, dropout_rate=0.2).to(device)

c:\Users\Admin\.conda\envs\ML_AIoT\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\.conda\envs\ML_AIoT\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## Initial Training (Frozen Base Layers)

In [5]:
print("Training with frozen base layers...")
optimizer = torch.optim.Adam(model.base_model.fc.parameters(), lr=0.001)  # Only train fc
criterion = nn.CrossEntropyLoss()

for epoch in range(NUM_EPOCHS_INITIAL):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print(f"Initial Epoch [{epoch+1}/{NUM_EPOCHS_INITIAL}], Batch [{i+1}], Loss: {running_loss / 100:.4f}")
            running_loss = 0.0
    
    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_accuracy = 100 * correct / total
    print(f"Initial Epoch [{epoch+1}/{NUM_EPOCHS_INITIAL}] Val Accuracy: {val_accuracy:.2f}%")

Training with frozen base layers...
Initial Epoch [1/10], Batch [100], Loss: 2.3714
Initial Epoch [1/10], Batch [200], Loss: 0.6524
Initial Epoch [1/10], Batch [300], Loss: 0.4711
Initial Epoch [1/10], Batch [400], Loss: 0.3836
Initial Epoch [1/10], Batch [500], Loss: 0.3329
Initial Epoch [1/10], Batch [600], Loss: 0.3641
Initial Epoch [1/10], Batch [700], Loss: 0.3096
Initial Epoch [1/10], Batch [800], Loss: 0.3312
Initial Epoch [1/10], Batch [900], Loss: 0.3248
Initial Epoch [1/10], Batch [1000], Loss: 0.3035
Initial Epoch [1/10], Batch [1100], Loss: 0.3017
Initial Epoch [1/10], Batch [1200], Loss: 0.3472
Initial Epoch [1/10] Val Accuracy: 92.94%
Initial Epoch [2/10], Batch [100], Loss: 0.2541
Initial Epoch [2/10], Batch [200], Loss: 0.2277
Initial Epoch [2/10], Batch [300], Loss: 0.2618
Initial Epoch [2/10], Batch [400], Loss: 0.2462
Initial Epoch [2/10], Batch [500], Loss: 0.2390
Initial Epoch [2/10], Batch [600], Loss: 0.2419
Initial Epoch [2/10], Batch [700], Loss: 0.2541
Initial

## Fine-Tuning (Unfreeze Some Layers)

In [6]:
print("\nFine-tuning with unfrozen layers...")
model.unfreeze_layers(num_layers_to_unfreeze=20)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=5, min_lr=1e-6)

best_val_accuracy = 0.0
for epoch in range(NUM_EPOCHS_FINE):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print(f"Fine-Tune Epoch [{epoch+1}/{NUM_EPOCHS_FINE}], Batch [{i+1}], Loss: {running_loss / 100:.4f}")
            running_loss = 0.0
    
    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_accuracy = 100 * correct / total
    print(f"Fine-Tune Epoch [{epoch+1}/{NUM_EPOCHS_FINE}] Val Accuracy: {val_accuracy:.2f}%")
    
    # Scheduler and save best model
    scheduler.step(val_accuracy)
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Saved best model with val accuracy {best_val_accuracy:.2f}%")

print("Fine-tuning completed!")


Fine-tuning with unfrozen layers...
Fine-Tune Epoch [1/50], Batch [100], Loss: 0.5049
Fine-Tune Epoch [1/50], Batch [200], Loss: 0.6541
Fine-Tune Epoch [1/50], Batch [300], Loss: 0.6943
Fine-Tune Epoch [1/50], Batch [400], Loss: 0.5846
Fine-Tune Epoch [1/50], Batch [500], Loss: 0.5094
Fine-Tune Epoch [1/50], Batch [600], Loss: 0.5609
Fine-Tune Epoch [1/50], Batch [700], Loss: 0.4521
Fine-Tune Epoch [1/50], Batch [800], Loss: 0.5294
Fine-Tune Epoch [1/50], Batch [900], Loss: 0.4774
Fine-Tune Epoch [1/50], Batch [1000], Loss: 0.4440
Fine-Tune Epoch [1/50], Batch [1100], Loss: 0.4880
Fine-Tune Epoch [1/50], Batch [1200], Loss: 0.4679
Fine-Tune Epoch [1/50] Val Accuracy: 87.76%
Saved best model with val accuracy 87.76%
Fine-Tune Epoch [2/50], Batch [100], Loss: 0.2357
Fine-Tune Epoch [2/50], Batch [200], Loss: 0.1933
Fine-Tune Epoch [2/50], Batch [300], Loss: 0.2032
Fine-Tune Epoch [2/50], Batch [400], Loss: 0.2535
Fine-Tune Epoch [2/50], Batch [500], Loss: 0.2383
Fine-Tune Epoch [2/50], 

## Final Evaluation

In [7]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
final_accuracy = 100 * correct / total
print(f"Final Validation Accuracy: {final_accuracy:.2f}%")

if final_accuracy >= 90.0:
    print(f"\nTarget accuracy (>90%) achieved! Saving final model to {SAVE_PATH}")
    torch.save(model.state_dict(), SAVE_PATH)
else:
    print(f"\nTarget accuracy (>90%) not achieved. Best model saved with {best_val_accuracy:.2f}%.")

Final Validation Accuracy: 90.58%

Target accuracy (>90%) achieved! Saving final model to D:\AIoT_project\models\resnet50_miniimagenet_90_v2.pth


## Load and Verify Model

In [8]:
print(f"\nLoading saved model from {SAVE_PATH} for verification...")
loaded_model = FineTuneResNet50(num_classes=num_classes, dropout_rate=0.2).to(device)
loaded_model.load_state_dict(torch.load(SAVE_PATH))
loaded_model.eval()

correct, total = 0, 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = loaded_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
loaded_accuracy = 100 * correct / total
print(f"Loaded Model Validation Accuracy: {loaded_accuracy:.2f}%")


Loading saved model from D:\AIoT_project\models\resnet50_miniimagenet_90_v2.pth for verification...
Loaded Model Validation Accuracy: 90.58%
